# libraries import #

In [1]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
import os
print(os.getcwd())

c:\project data\fyp


In [3]:
import os
print(os.listdir())

['.vscode', 'advanced_app.py', 'again model train.ipynb', 'app.py', 'class_distribution.png', 'confusion_matrix.png', 'fake job.csv', 'fake_job.ipynb', 'history', 'metrics.png', 'models', 'phishing_detector.pkl', 'Phishing_Email.csv', 'requirements.txt', 'requirements_2.txt', 'run_project.bat', 'sms spam.csv', 'sms_spam_model.pkl', 'sms_vectorizer.pkl', 'train_sms_model.ipynb']


# dataset path #

In [18]:
CSV_PATH = r"Phishing_Email.csv"
CHUNK_SIZE = 3000

TEXT_COL = "Email Text"
LABEL_COL = "Email Type"

# Analyze dataset structure #

In [20]:
sample = pd.read_csv(CSV_PATH, nrows=5)

print("Columns:")
print(sample.columns)

print("\nFirst 5 Rows:")
print(sample.head())

Columns:
Index(['ru', 'Email Text', 'Email Type'], dtype='object')

First 5 Rows:
   ru                                         Email Text      Email Type
0   0  re : 6 . 1100 , disc : uniformitarianism , re ...      Safe Email
1   1  the other side of * galicismos * * galicismo *...      Safe Email
2   2  re : equistar deal tickets are you still avail...      Safe Email
3   3  \nHello I am your hot lil horny toy.\n    I am...  Phishing Email
4   4  software at incredibly low prices ( 86 % lower...  Phishing Email


# text cleaning function #

In [21]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"www\S+", " ", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# label encoding #

In [22]:
le = LabelEncoder()

le.fit([
    "Phishing Email",
    "Safe Email"
])

classes = np.arange(
    len(le.classes_)
)

valid_labels = set(le.classes_)

# feature extraction #

In [23]:
vectorizer = HashingVectorizer(
    n_features=2**18,
    alternate_sign=False,
    stop_words="english"
)

# ML training #

In [24]:
model = SGDClassifier(
    loss="log_loss",
    random_state=42
)

# Training Generator #

In [25]:
def data_generator():

    for chunk in pd.read_csv(
        CSV_PATH,
        chunksize=CHUNK_SIZE,
        usecols=[TEXT_COL, LABEL_COL]
    ):

        # Remove missing values
        chunk = chunk.dropna()

        # Remove duplicates
        chunk = chunk.drop_duplicates()

        # Remove empty emails
        chunk = chunk[
            chunk[TEXT_COL].astype(str).str.strip() != ""
        ]
        chunk = chunk[chunk[LABEL_COL].isin(valid_labels)]

        if len(chunk) == 0:
            continue

        chunk[TEXT_COL] = (
            chunk[TEXT_COL]
            .astype(str)
            .apply(clean_text)
        )

        X = vectorizer.transform(
            chunk[TEXT_COL]
        )
        y = le.transform(
            chunk[LABEL_COL]
        )

        X_train,X_test,y_train,y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )
        yield X_train,X_test,y_train,y_test

# Train Model #

In [26]:
first_chunk = True

for X_train,X_test,y_train,y_test in data_generator():

    if first_chunk:

        model.partial_fit(
            X_train,
            y_train,
            classes=classes
        )

        first_chunk = False

    else:

        model.partial_fit(X_train,y_train)

print("Training Complete")

Training Complete


# Evalution #

In [27]:
y_true = []
y_pred = []

for X_train,X_test,y_train,y_test in data_generator():

    predictions = model.predict(X_test)

    y_true.extend(y_test)

    y_pred.extend(predictions)

# metrices #

In [28]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

print("\nAccuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)


Accuracy : 0.9691666666666666
Precision: 0.9626933575978162
Recall   : 0.9864801864801864
F1 Score : 0.9744416302095326


# confusion matrix #

In [29]:
cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix")
print(cm)


Confusion Matrix
[[1373   82]
 [  29 2116]]


# classification reports #

In [30]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=le.classes_
    )
)

                precision    recall  f1-score   support

Phishing Email       0.98      0.94      0.96      1455
    Safe Email       0.96      0.99      0.97      2145

      accuracy                           0.97      3600
     macro avg       0.97      0.97      0.97      3600
  weighted avg       0.97      0.97      0.97      3600



# saved model #

In [31]:
with open(
    "phishing_detector.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "model": model,
            "vectorizer": vectorizer,
            "encoder": le
        },
        f
    )

print("Model Saved")

Model Saved


# test email #

In [32]:
sample = [
    "Congratulations you won a free iphone click now"
]

sample = [
    clean_text(x)
    for x in sample
]

X_test = vectorizer.transform(sample)

prediction = model.predict(X_test)

print(
    le.inverse_transform(prediction)[0]
)

Phishing Email
